# BN Inference (BoNesis)

Continues from **"5. GRN Construction.ipynb"**: loads the exported PKN (`{FILE_PREFIX}_prior_grn.json` / `{FILE_PREFIX}_pkn.json`), the per-phenotype observations (`{FILE_PREFIX}_data_meta.json`) and the run config (`{FILE_PREFIX}_config.json`), then runs the actual BoNesis ASP solving to infer a Boolean Network consistent with the observed phenotype states.

Set `FILE_PREFIX` below to the same value used in "5. GRN Construction.ipynb" (`"sum"`, `"mode"` or `"bootstrap"`).

The dynamical constraints are **not hardcoded here**: they come from `transition_constraints.json` written by the trajectory step (`4. Trajectory Inference.ipynb` / `4. RNA Velocity.ipynb`, set `CONSTRAINTS_JSON` below) and are declared with `apply_bonesis_constraints` (rules: `src/trajectory_inference/README.md`). Every state they mention must be an observation in `data_meta`, i.e. the observation tags used in "4.5. Phenotype Observations.ipynb" must come from the same trajectory run.

In [ ]:
# Must match the FILE_PREFIX used when exporting from "5. GRN Construction.ipynb"
FILE_PREFIX = "sum"

# transition_constraints.json from the trajectory step (same run as the observation tags used in 4.5.)
CONSTRAINTS_JSON = "results/paga_dpt/transition_constraints.json"
# False = only reachability + fixed points (paper's speed-up for node selection; Chevalier et al. 2025)
NEGATIVE_CONSTRAINTS = True

In [ ]:
import sys
import json
import datetime
from pathlib import Path as path

import networkx as nx
import bonesis

from scboolseq.utils import Timer
from markers import marker_genes

# Repo root → src on path (shared with the RNA velocity / trajectory-inference notebooks)
ROOT = path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from grn_inference import load_pkn_bundle, extract_largest_scc
from trajectory_inference import apply_bonesis_constraints


In [ ]:
prior_grn, pkn, data_meta, _ = load_pkn_bundle(FILE_PREFIX)
with open(CONSTRAINTS_JSON) as _f:
    constraints = json.load(_f)

print(
    f"FILE_PREFIX={FILE_PREFIX!r}, prior_grn nodes={len(prior_grn)}, pkn nodes={len(pkn)}, "
    f"observations={len(data_meta)}"
)
print(f"constraints from {CONSTRAINTS_JSON} (method={constraints['method']!r}, edges from {constraints['edge_source']!r}):")
print("\n".join(constraints["bonesis_lines"]))

In [ ]:
bo = bonesis.BoNesis(pkn, data_meta)
bo


In [ ]:
apply_bonesis_constraints(bo, constraints, negative=NEGATIVE_CONSTRAINTS)

In [ ]:
date = datetime.datetime.now()
bo.maximize_nodes()
bo.maximize_strong_constants()
view = bonesis.NonStrongConstantNodesView(bo, mode="optN")
view.standalone(
    output_filename=f"optim_dorothea_v2_{FILE_PREFIX}_{date.strftime('%Y-%m-%d')}.sh"
)
print("Standalone shell file written", flush=True)

kept_nodes = {}
with Timer("Optimizing NonStrongConstantNodesView(bo, mode='optN')"):
    for kept_nodes in view:
        break

_replica_grn = nx.subgraph(prior_grn, kept_nodes)


In [ ]:
len(kept_nodes)


In [ ]:
marker_genes.isin(_replica_grn).mean()


In [ ]:
bonesis.InfluenceGraph(_replica_grn)


In [ ]:
EXPORT = True

if EXPORT:
    nx.nx_pydot.write_dot(extract_largest_scc(bonesis.InfluenceGraph(_replica_grn)), f"{FILE_PREFIX}_grn_scc_v1.dot")
    with open(f"{FILE_PREFIX}_grn_optimized_v1.json", "w") as f:
        json.dump(nx.node_link_data(_replica_grn), f)

    with open(f"{FILE_PREFIX}_pkn_bonesis_obj_optimized_v1.json", "w") as f:
        json.dump(nx.node_link_data(bonesis.InfluenceGraph(_replica_grn)), f)
    print("Exported")
else:
    print("Not exported")
